In [2]:
import pandas as pd
import re

In [3]:
msg = pd.read_csv('./data/MSG.CSV', index_col=0)
msg['STND_YMD'] = pd.to_datetime(msg['STND_YMD'])
print(msg.shape)
msg.head(2)

(4931532, 22)


,STND_YMD,INCS_NO,MSG_SEND_CHN_CD,MSG_SEND_CHN_NM,MSG_SEND_OFR_CHN_CD,MSG_SEND_OFR_CHN_NM,JOUR_ID,JOUR_NM,CAMP_TGT_CHN_CD,CAMP_TGT_CHN_NM,...,MSG_SEND_CNT,ETL_PROC_DTTM,JOUR,저니,저니명,버전,발송타입,발송시간,본문,버튼
0,2024-10-28,f3f306c20e5caa497df4bbbd4732161eb63f65b92544b3...,FT,카카오 친구톡,IN,이니스프리,5EEB341B-52BF-4707-8B40-C6E811D725B5,IN24102704S_IF_2410_5주차_LEE4554,"[\n ""036""\n]","[\n ""이니스프리 쇼핑몰""\n]",...,1,2025-01-20 04:46:47.591,IN24102704S,IN24102704S,IN24102704S_IF_2410_5주차_LEE4554,2,친구톡,2024-10-28 11:00:00,"이니스프리 | Lee 토트백 소진 임박 (우와)\n\n· 25,000원 구매 시, ...","{""button"":[{""name"":""구경 가기▶"", ""type"":""AL"", ""sch..."
1,2024-09-08,906522474e459ca44336c858e76b7f6cc4a8b76abba594...,FT,카카오 친구톡,IN,이니스프리,1AFE739B-45FD-44AB-8206-0D4A29641558,IN24090411S_IF_2409_레티놀패드_크림,"[\n ""036""\n]","[\n ""이니스프리 쇼핑몰""\n]",...,1,2025-01-20 04:46:47.591,IN24090411S,IN24090411S,IN24090411S_IF_2409_레티놀패드_크림,2,친구톡,2024-09-08 11:00:00,[33% 할인] 레티놀 러버(하트) 희소식!\n지금 사면 패드 2개가 3만원 초반대...,"{""button"":[{""name"":""구경 가기▶"", ""type"":""AL"", ""sch..."


### 메세지 indicating
* 메시지 길이
* 할인율
    * 여러개의 경우 가장 할인율이 높은 수치
    * 50% -> 50
    * 1+1 -> 50 / 2+1 -> 33
* 이모지 수
    * (하트), (우와)
* 시간압박
    * D-0
    * SOLD OUT
    * 소진 임박
* 개인화 여부
    * 00고객님, 00님

In [4]:
temp = list(set(msg['본문']))
len(temp)

230

In [5]:
msg_length_list = []
max_discount_list = []
emoji_count_list = []
time_pressure_list = []
personal_list = []

for idx, row in msg.iterrows():
    text = row['본문']

    # 메시지 길이
    msg_length_list.append(len(row['본문']))
    
    # 할인율
    max_discount = 0  # 기본 할인율 0으로 설정
    discount_matches = re.findall(r'(\d+)%', text)
    bogo_matches = re.findall(r'(\d+)\+(\d+)', text)
        
    if discount_matches:
        max_discount = max(map(int, discount_matches))
        
    for buy, free in bogo_matches:
        discount = int(free) / (int(buy) + int(free)) * 100
        if discount > max_discount:
            max_discount = discount
    max_discount_list.append(max_discount)


    personal_list.append(int(bool(re.search(r'님', text))))

    # 이모지 수
    emoji_count = text.count('(하트)') + text.count('(우와)')
    emoji_count_list.append(emoji_count)

    # 시간압박 여부
    time_pressure = int(bool(re.search(r'D-|SOLD OUT|소진 임박', text)))
    time_pressure_list.append(time_pressure)

In [9]:
msg_indicate = pd.DataFrame(
    {   
        'STND_YMD' : msg['STND_YMD'],
        'INCS_NO' : msg['INCS_NO'],
        'msg_length' : msg_length_list,
        'max_discount' : max_discount_list,
        'emoji_count' : emoji_count_list,
        'personalized' : personal_list,
        'time_pressure' : time_pressure_list,
    }
)
msg_indicate.head(2)

,STND_YMD,INCS_NO,msg_length,max_discount,emoji_count,personalized,time_pressure
0,2024-10-28,f3f306c20e5caa497df4bbbd4732161eb63f65b92544b3...,76,64.0,1,1,1
1,2024-09-08,906522474e459ca44336c858e76b7f6cc4a8b76abba594...,49,33.0,2,0,0


In [14]:
msg_indicate.to_csv('./data/msg_indicate.csv')